以下为对Neurex模型内容的检查

In [61]:
import torch
import os

def inspect_pytorch_model(model_path):
    """
    检查pytorch_model.bin文件中的权重名称
    
    Args:
        model_path: 模型文件所在的目录路径
    """
    # 加载模型权重
    checkpoint_path = os.path.join(model_path, "pytorch_model.bin")
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    
    # 打印所有权重名称
    print("模型权重结构:")
    print("=" * 50)
    
    # 创建前缀分组
    prefixes = {}
    for key in state_dict.keys():
        # 获取前缀 (通常是模型层的名称)
        prefix = key.split('.')[0]
        if prefix not in prefixes:
            prefixes[prefix] = []
        prefixes[prefix].append(key)
    
    # 打印每个分组的前10个参数
    for prefix, keys in prefixes.items():
        print(f"\n[{prefix}]: {len(keys)} parameters")
        print("-" * 40)
        for i, key in enumerate(keys[:10]):  # 只打印前10个权重
            shape = state_dict[key].shape
            print(f"{key}: {shape}")
        
        if len(keys) > 10:
            print(f"... 和其他 {len(keys) - 10} 个参数")
    
    # 特别查找可能用于分类的权重
    print("\n可能的分类器权重:")
    print("=" * 50)
    classifier_keywords = ["classifier", "head", "cls", "output", "predict", "fc"]
    for key in state_dict.keys():
        for keyword in classifier_keywords:
            if keyword in key.lower():
                shape = state_dict[key].shape
                print(f"{key}: {shape}")
                break

    # 打印隐藏层大小信息(用于确定特征向量维度)
    embedding_keys = [k for k in state_dict.keys() if "embeddings" in k and "weight" in k]
    if embedding_keys:
        embed_key = embedding_keys[0]
        print(f"\n嵌入层大小: {state_dict[embed_key].shape}")
    
    # 查找最后一层的维度
    output_keys = [k for k in state_dict.keys() if "output" in k.lower() and "weight" in k]
    if output_keys:
        for key in output_keys:
            print(f"输出层: {key} - {state_dict[key].shape}")

model_path = "checkpoint-30768"  # 替换为你的模型路径
inspect_pytorch_model(model_path)

模型权重结构:

[roberta]: 198 parameters
----------------------------------------
roberta.embeddings.position_ids: torch.Size([1, 514])
roberta.embeddings.word_embeddings.weight: torch.Size([50265, 768])
roberta.embeddings.position_embeddings.weight: torch.Size([514, 768])
roberta.embeddings.token_type_embeddings.weight: torch.Size([1, 768])
roberta.embeddings.LayerNorm.weight: torch.Size([768])
roberta.embeddings.LayerNorm.bias: torch.Size([768])
roberta.encoder.layer.0.attention.self.query.weight: torch.Size([768, 768])
roberta.encoder.layer.0.attention.self.query.bias: torch.Size([768])
roberta.encoder.layer.0.attention.self.key.weight: torch.Size([768, 768])
roberta.encoder.layer.0.attention.self.key.bias: torch.Size([768])
... 和其他 188 个参数

[token_classifier]: 2 parameters
----------------------------------------
token_classifier.weight: torch.Size([3, 768])
token_classifier.bias: torch.Size([3])

[cls_classifier]: 2 parameters
----------------------------------------
cls_classifier.weig

以下为用于模型加载的函数定义 （：：：：start）

In [70]:
from transformers import RobertaModel, RobertaTokenizer
import torch
import os

class CodeBERTClassifier:
    def __init__(self, model_path):
        """
        初始化CodeBERT模型和分类器
        
        Args:
            model_path: 模型文件所在的目录路径
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 加载tokenizer
        self.tokenizer = RobertaTokenizer.from_pretrained(model_path)

        # 获取特殊token
        self.cls_token = self.tokenizer.cls_token  # 通常是[CLS]
        self.sep_token = self.tokenizer.sep_token  # 通常是[SEP]
        
        # 加载基础模型 - 用于提取特征向量
        self.model = RobertaModel.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        
        # 加载二分类器
        self.binary_classifier = self._load_cls_classifier(model_path)
        
    def _load_cls_classifier(self, model_path):
        """
        加载CLS分类器
        """
        # 加载完整的模型权重
        state_dict_path = os.path.join(model_path, "pytorch_model.bin")
        state_dict = torch.load(state_dict_path, map_location=self.device)
        
        # 创建一个简单的线性分类器
        hidden_size = 768  # 根据模型规格，通常是768
        classifier = torch.nn.Linear(hidden_size, 2, bias=False)  # 二分类器，没有偏置
        
        # 查找并加载cls_classifier的权重
        if 'cls_classifier.weight' in state_dict:
            # 直接加载权重到分类器
            classifier.weight.data = state_dict['cls_classifier.weight']
            print("成功加载cls_classifier权重")
        else:
            print("警告：未找到cls_classifier.weight，使用随机初始化")
        
        # 如果存在偏置项，也加载它
        if 'cls_classifier.bias' in state_dict:
            classifier.bias = torch.nn.Parameter(state_dict['cls_classifier.bias'])
            print("成功加载cls_classifier偏置")
        
        return classifier.to(self.device)
    
    def get_cls_vector(self, text):
        """
        提取输入文本的[CLS]向量表示
        
        Args:
            text: 输入文本字符串
            
        Returns:
            tensor: [CLS]的向量表示
        """
        # 对输入文本进行编码
        processed_text = self.preprocess_text(text)
        # processed_text = text
        inputs = self.tokenizer(processed_text, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # 提取特征向量
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # 获取[CLS]向量 - 最后一层的第一个token
        last_hidden_state = outputs.last_hidden_state
        cls_vector = last_hidden_state[:, 0, :]  # 取第一个位置([CLS])的向量
        
        return cls_vector
    
    def preprocess_text(self, text):
        """
        预处理文本：添加[CLS]并替换换行符为[SEP]
        """
        # 在开头添加[CLS]
        processed_text = f"{self.cls_token} {text}"
        
        # 将所有换行符替换为[SEP]
        processed_text = processed_text.replace('\n', f' {self.sep_token} ')
        
        return processed_text
    
    def classify(self, text):
        """
        对输入文本进行二分类
        
        Args:
            text: 输入文本字符串
            
        Returns:
            dict: 包含预测标签和概率的字典
        """
        # 获取[CLS]向量表示
        cls_vector = self.get_cls_vector(text)
        
        # 使用分类器进行预测
        with torch.no_grad():
            logits = self.binary_classifier(cls_vector)
            probabilities = torch.softmax(logits, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            probability = probabilities[0][predicted_class].item()
        
        return {
            "label": predicted_class,
            "probability": probability,
            "probabilities": probabilities[0].tolist()  # 返回所有类别的概率
        }

def test_model(model_path, text):
    """
    测试模型函数
    """
    # 初始化分类器
    classifier = CodeBERTClassifier(model_path)
    
    # 进行预测
    result = classifier.classify(text)
    
    # 打印结果
    print(f"输入文本: {text}")
    print(f"预测标签: {result['label']}")
    print(f"预测概率: {result['probability']:.4f}")
    print(f"各类别概率: {[f'{p:.4f}' for p in result['probabilities']]}")
    
    return result

以下加载模型并进行预测

In [71]:
model_path = "checkpoint-30768"
classifier = CodeBERTClassifier(model_path)

Some weights of RobertaModel were not initialized from the model checkpoint at checkpoint-30768 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


成功加载cls_classifier权重
成功加载cls_classifier偏置


In [72]:
# 进行预测
text = "def hello_world():\n    print('Hello, World!')"
result = classifier.classify(text)
print(result)

{'label': 0, 'probability': 0.9988127946853638, 'probabilities': [0.9988127946853638, 0.0011871707392856479]}


In [ ]:
import json

input_file = "../data/experiment_result_cleanE0401.json"
target_file = "experiment_result_clean-bl1E0401-rerun.json"
experiment_data = []
with open(input_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

model_path = "checkpoint-30768"
classifier = CodeBERTClassifier(model_path)

target_data = []
for item in experiment_data:
    # 获取方法内容
    method = item.get("methodBefore")

    result = classifier.classify(method)

    item["changed"] = result['label']
    print(str(result['label']) + ', ' , end=None)

    target_data.append(item)

with open(target_file, "w", encoding="utf-8") as f:
    json.dump(target_data, f, ensure_ascii=False, indent=4)  # 格式化输出

Some weights of RobertaModel were not initialized from the model checkpoint at checkpoint-30768 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


成功加载cls_classifier权重
成功加载cls_classifier偏置
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
1, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
1, 
0, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
1, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0, 
0,

以下为数据计算

In [ ]:
import json

target_file = "experiment_result_clean-bl1E0401.json"
experiment_data = []
with open(target_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_metrics(experiment_data):
    # 初始化变量
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    
    for item in experiment_data:
        # 获取标签和预测值
        true_label = item.get("label") 
        predicted_value = item.get("changed")
        
        # # 跳过需要舍弃的数据
        # if true_label == -1:
        #     continue

        # if item.get("methodBefore").count('\n') < 8:
        #     continue

        # if item.get("methodBefore").startswith("@Test") or item.get("file_path").endswith("Test.java"):
        #     continue

        # if len(item.get("exceptionTypes")) == 1 and "RuntimeException" in item.get("exceptionTypes"):
        #     continue
        
        # 确定真实类别和预测类别
        true_class = 1 if true_label == 1 else 0
        predicted_class = 1 if predicted_value > 0 else 0
        
        # 更新统计量
        if true_class == 1 and predicted_class == 1:
            true_positives += 1
        elif true_class == 1 and predicted_class == 0:
            false_negatives += 1
        elif true_class == 0 and predicted_class == 1:
            false_positives += 1
        elif true_class == 0 and predicted_class == 0:
            true_negatives += 1
    
    # 计算指标
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "total": true_positives + false_positives + false_negatives + true_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "true_negatives": true_negatives
    }


metrics = calculate_metrics(experiment_data)
print(metrics)
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")

{'total': 1604, 'precision': 0.5, 'recall': 0.03148148148148148, 'f1': 0.059233449477351915, 'true_positives': 17, 'false_positives': 17, 'false_negatives': 523, 'true_negatives': 1047}
Precision: 0.5000
Recall: 0.0315
F1 Score: 0.0592
